In [ ]:
%pip install python-dotenv
%pip install roboflow
%pip install supervision

In [ ]:
# loads dataset
from roboflow import Roboflow
from dotenv import load_dotenv
import os

load_dotenv()  # loads variables from .env into the environment

api_key = os.getenv("YF_API_KEY")

rf = Roboflow(api_key=api_key)
project = rf.workspace("caretech").project("food-dataset-uj20h-w2s4m")
version = project.version(1)
dataset = version.download("yolov8")

In [ ]:
%pip install ipywidgets

In [ ]:
# script to split training dataset
import os
import shutil
from pathlib import Path
import supervision as sv


# creates a new folder with just the images and labels
dir = Path('/content/Food-Dataset-1')
unified_dir = Path('/content/all')
split_path = Path('/content/split')

new_img_dir = unified_dir / 'images'
new_label_dir = unified_dir / 'labels'

new_img_dir.mkdir(parents=True, exist_ok=True)
new_label_dir.mkdir(parents=True, exist_ok=True)

# write to a flattened folder
for file in dir.rglob('*.jpg'):
    shutil.copy(file, new_img_dir)

exclude = ['README.roboflow.txt', 'README.dataset.txt']
for file in dir.rglob('*.txt'):
    if str(file.name) in exclude:
        continue
    shutil.copy(file, new_label_dir)


In [ ]:
# this loads a DetectionDataset object
ds = sv.DetectionDataset.from_yolo(
    images_directory_path=str(new_img_dir),
    annotations_directory_path=str(new_label_dir),
    data_yaml_path=str(dir / 'data.yaml')
)

print(ds.classes)

seed = 1
# we can split this dataset deterministically
train_ds, rest_ds = ds.split(split_ratio=0.8, random_state=seed, shuffle=True)
test_ds, val_ds = rest_ds.split(split_ratio=0.5, random_state=seed, shuffle=True)

# save new datasets in yolo format
train_ds.as_yolo(
    images_directory_path=str(split_path / 'train' / 'images'),
    annotations_directory_path=str(split_path / 'train' / 'labels'),
    data_yaml_path=str(split_path / 'train' / 'data.yaml')
)
test_ds.as_yolo(
    images_directory_path=str(split_path / 'test' / 'images'),
    annotations_directory_path=str(split_path / 'test' / 'labels'),
    data_yaml_path=str(split_path / 'test' / 'data.yaml')
)
val_ds.as_yolo(
    images_directory_path=str(split_path / 'valid' / 'images'),
    annotations_directory_path=str(split_path / 'valid' / 'labels'),
    data_yaml_path=str(split_path / 'valid' / 'data.yaml')
)

# write the manifest files
def write_manifest(ds: sv.DetectionDataset, split: str, output_path: Path):
  with open(output_path, "w") as f:
    for img_path, _, _ in ds:
      f.write(f"{str(Path(img_path).name)}\n")

  print(f"Wrote to {output_path}")

write_manifest(train_ds, "train", Path(split_path) / "train.txt")
write_manifest(test_ds, "test", Path(split_path) / "test.txt")
write_manifest(val_ds, "valid", Path(split_path) / "valid.txt")

In [ ]:
# class distribution count
from collections import defaultdict
from matplotlib import pyplot as plt
from matplotlib.ticker import MaxNLocator

class_labels = { i: ds.classes[i] for i in range(len(ds.classes)) }

class_count = defaultdict(int)
box_count = defaultdict(int)
label_dir = unified_dir / 'labels'
label_files = list(label_dir.glob('*.txt'))
print(f"Label files found: {len(label_files)}")
for file in label_files:
    count = 0
    with open(file, 'r') as f:
      for line in f:
        arr = line.split()
        if not arr:
          continue

        i = int(arr[0])
        class_count[class_labels[i]] += 1
        count += 1
      

    box_count[count] += 1


print(class_count)

# class distribution count
# image size
# aspect ratio
# boxes per image
# noise / blur

# plot
plt.figure(figsize=(10, 6))
plt.bar(class_count.keys(), class_count.values())
plt.xlabel('Class')
plt.ylabel('Count')
plt.title('Class Distribution')
plt.xticks(rotation=90)
plt.show()

# plot
plt.figure(figsize=(10, 6))
ax = plt.figure().gca()
plt.bar(box_count.keys(), box_count.values())
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.set_xlim([0, 10])
plt.xlabel('Box Count')
plt.ylabel('Count')
plt.title('Box Count Distribution')
plt.show()

In [ ]:
from PIL import Image

# image size
sizes: list[tuple[int, int]] = [] # width x height
img_dir = unified_dir / 'images'
images = list(img_dir.glob('*.jpg'))
print(f"Images found: {len(images)}")
for img in images:
  with Image.open(img) as im:
    sizes.append(im.size)

ratios = [s[0] / s[1] for s in sizes]

# plot
plt.figure(figsize=(10, 6))
plt.scatter([s[0] for s in sizes], [s[1] for s in sizes])
plt.xlabel('Width')
plt.ylabel('Height')
plt.title('Image Sizes')
plt.show()

plt.figure(figsize=(10, 6))
plt.hist(ratios, bins=100, range=(0.5, 2.5))
plt.xlabel('Aspect Ratio')
plt.ylabel('Count')
plt.title('Aspect Ratio Distribution')
plt.show()

In [ ]:
import glob

for root, dirs, files in os.walk("/content", topdown=True):
    if os.path.basename(root) == "images":
        print("IMAGES FOLDER:", root)
        imgs = glob.glob(os.path.join(root, "*.jpg"))
        print("IMAGES FOLDER:", root, "->", len(imgs))


In [ ]:
%pip install ultralytics

In [ ]:
from pathlib import Path
from collections import Counter

def count_images_per_class(labels_dir, class_names):
    labels_dir = Path(labels_dir)
    image_count_per_class = Counter()
    
    for lbl_file in labels_dir.glob('*.txt'):
        with lbl_file.open() as f:
            classes_in_image = set(int(line.split()[0]) for line in f if line.strip())
        for cls_id in classes_in_image:
            image_count_per_class[cls_id] += 1
    
    print(f"\nClass distribution in {labels_dir.parent.name}:")

    for cls_id in range(len(class_names)):
        count = image_count_per_class.get(cls_id, 0)
        print(f"{cls_id:2d} {class_names[cls_id]:30s}: {count:4d} images")

    return image_count_per_class

train_labels = split_path / 'train' / 'labels'
valid_labels = split_path / 'valid' / 'labels'
test_labels = split_path / 'test' / 'labels'

class_names = ds.classes

train_counts = count_images_per_class(train_labels, class_names)
valid_counts = count_images_per_class(valid_labels, class_names)
test_counts = count_images_per_class(test_labels, class_names)

In [ ]:
import yaml

data_yaml_content = {
    'path': str(split_path.resolve()), 
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': len(ds.classes),
    'names': ds.classes
}

unified_yaml_path = split_path / 'data.yaml'
with open(unified_yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f, sort_keys=False)

print(f"Created unified data.yaml at: {unified_yaml_path}")
print(f"   Number of classes: {len(ds.classes)}")

In [ ]:
from ultralytics import YOLO


model = YOLO('yolov8n.pt')  

training_config = {
    'data': str(unified_yaml_path),        
    'epochs': 50,                          
    'imgsz': 640,                          
    'batch': 16,                           
    'patience': 10,                        
    'save': True,                          
    'close_mosaic': 10,                    
    'cos_lr': True,                        
    'optimizer': 'AdamW',                  # (AdamW, SGD, Adam)
    'lr0': 0.001,                          
    'weight_decay': 0.0005,                
}

In [ ]:
results = model.train(**training_config)

print("Training done")

In [ ]:
print("Running validation")

best_model = YOLO('runs/detect/food_yolov8n/weights/best.pt')
val_results = best_model.val(data=str(unified_yaml_path))

print("Validation Results:")

print(f"  mAP@0.5      : {val_results.box.map50:.4f}")
print(f"  mAP@0.5:0.95 : {val_results.box.map:.4f}")
print(f"  Precision    : {val_results.box.mp:.4f}")
print(f"  Recall       : {val_results.box.mr:.4f}")


In [ ]:

import random

test_images = list((split_path / 'test' / 'images').glob('*.jpg'))
sample_images = random.sample(test_images, min(3, len(test_images)))

print("Testing inference on sample images from test set:")

for img_path in sample_images:
    print(f"Processing: {img_path.name}")
    

    results = best_model.predict(
        source=str(img_path),
        conf=0.25,      
        iou=0.5,        
    )
    
    for r in results:
        print(f"  Detected {len(r.boxes)} objects")
        for box in r.boxes:
            cls_id = int(box.cls[0])
            conf = float(box.conf[0])
            print(f"    - {class_names[cls_id]}: {conf:.2f}")
